# Métricas con Prometheus

Instrumentar tu app con métricas Prometheus

## Introducción

Prometheus recolecta principalmente series temporales numéricas etiquetadas. Cada métrica tiene un nombre, etiquetas (labels) key-value, y valores numéricos capturados en timestamps específicos.

### Objetivos de Aprendizaje

- Entender qué tipo de datos recolecta Prometheus
- Conocer los tipos de métricas disponibles
- Instrumentar una aplicación Python con prometheus_client
- Definir labels para filtrado y segmentación
- Crear dashboards básicos en Grafana

## Qué tipo de dato recolecta Prometheus

> Prometheus recolecta series temporales numéricas etiquetadas. Cada punto de datos tiene: nombre de métrica, labels (clave=valor), valor numérico, y timestamp. Los labels permiten filtrar y agregar datos.

In [ ]:
print("=== Datos que recolecta Prometheus ===")

print("""
Series temporales numéricas etiquetadas:

http_requests_total{method="GET", status="200", handler="/api/users"} 15234
http_requests_total{method="POST", status="201", handler="/api/users"} 421
http_requests_total{method="GET", status="404", handler="/api/users"} 89

Partes de cada entrada:
  - http_requests_total  → nombre de la métrica
  - method, status, handler → labels (clave=valor)
  - 15234               → valor numérico (contador)
  - (implícito) timestamp → momento de la lectura

Esto permite consultas como:
  - Sumar todas las requests GET: sum(http_requests_total{method="GET"})
  - Filtrar por status 500: http_requests_total{status="500"}
  - Rate de requests por segundo: rate(http_requests_total[5m])
""")

## Tipos de Métricas de Prometheus

> Prometheus soporta cuatro tipos de métricas: Counter (acumulador), Gauge (valor que sube y baja), Histogram (distribución de valores), y Summary (percentiles).

In [ ]:
print("=== Tipos de Métricas ===")

metric_types = {
    "Counter": {
        "Descripción": "Solo sube, nunca baja. Para contar requests, errores, ventas.",
        "Ejemplo": "api_requests_total{status='200'}",
        "Uso": "¿Cuántas veces ocurrió X?"
    },
    "Gauge": {
        "Descripción": "Puede subir y bajar. Para valores instantáneos.",
        "Ejemplo": "memory_usage_bytes{region='us-east'}",
        "Uso": "¿Cuál es el valor actual de X?"
    },
    "Histogram": {
        "Descripción": "Observa distribución de valores en buckets.",
        "Ejemplo": "http_request_duration_seconds{le='0.1'}",
        "Uso": "¿Cuánto tiempo tardan las requests?"
    },
    "Summary": {
        "Descripción": "Calcula percentiles directamente.",
        "Ejemplo": "request_latency_seconds{quantile='0.95'}",
        "Uso": "¿Cuál es el p95 de latency?"
    },
}

for tipo, info in metric_types.items():
    print(f"\n{tipo}:")
    for k, v in info.items():
        print(f"  {k}: {v}")

## Instrumentar Aplicación Python

> Usa la librería prometheus_client para exponer métricas en tu aplicación FastAPI o Flask. Las métricas se exponen en /metrics para que Prometheus las scrapee.

In [ ]:
print("=== Instrumentación con prometheus_client ===")

instrumentation_code = """
# pip install prometheus-client

from prometheus_client import Counter, Histogram, Gauge, generate_latest
from fastapi import FastAPI, Response

app = FastAPI()

# Counter: cuenta requests
REQUEST_COUNT = Counter(
    'http_requests_total',
    'Total de HTTP requests',
    ['method', 'endpoint', 'status']
)

# Histogram: distribución de latencia
REQUEST_LATENCY = Histogram(
    'http_request_duration_seconds',
    'Latencia de HTTP requests',
    ['method', 'endpoint'],
    buckets=(0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0)
)

# Gauge: conexiones activas
ACTIVE_CONNECTIONS = Gauge(
    'active_connections',
    'Número de conexiones activas',
    ['service']
)

@app.get('/api/users')
async def get_users():
    REQUEST_COUNT.labels(method='GET', endpoint='/api/users', status='200').inc()
    with REQUEST_LATENCY.labels(method='GET', endpoint='/api/users').time():
        # Tu lógica de negocio
        return [{"id": 1, "name": "Juan"}]

@app.get('/metrics')
async def metrics():
    return Response(content=generate_latest(), media_type='text/plain')
"""

print(instrumentation_code)

## Labels para Filtrado y Segmentación

> Los labels son clave para segmentar métricas. Con labels puedes filtrar por servicio, región, versión, o cualquier dimensión que necesites para debugging y análisis.

In [ ]:
print("=== Uso de Labels ===")

print("""
Sin labels (una métrica para todo):
  api_requests_total = 50000

Con labels (segmentado):
  api_requests_total{method='GET', status='200'} = 45000
  api_requests_total{method='POST', status='201'} = 4000
  api_requests_total{method='GET', status='500'} = 1000

Consultas PromQL posibles:
  - requests por método: sum by (method) (api_requests_total)
  - Tasa de errores: rate(api_requests_total{status=~'5..'}[5m])
  - Requests por endpoint y status: sum by (endpoint, status) (api_requests_total)
""")

print("Buenos labels para API:")
buenos_labels = {
    "method": "GET, POST, PUT, DELETE",
    "endpoint": "/api/users, /api/products",
    "status": "200, 404, 500",
    "service": "auth, billing, inventory",
    "version": "v1, v2",
}
for label, values in buenos_labels.items():
    print(f"  {label}: {values}")

## Configuración de Prometheus para Scrape

> Prometheus scrapea endpoints /metrics de las aplicaciones. Se configura con un archivo prometheus.yml donde se define el targets y el intervalo de scrapeo.

In [ ]:
prometheus_yml = """
# prometheus.yml

global:
  scrape_interval: 15s
  evaluation_interval: 15s

scrape_configs:
  - job_name: 'mi-api'
    static_configs:
      - targets: ['localhost:8000']
    metrics_path: '/metrics'
    scrape_interval: 10s

  - job_name: 'mi-otro-servicio'
    static_configs:
      - targets: ['server-02:8080']

# Docker Compose integración:
  - job_name: 'docker'
    static_configs:
      - targets: ['host.docker.internal:9323']
"""

print("=== Configuración Prometheus ===")
print(prometheus_yml)

print("\nConceptos clave:")
conceptos = {
    "scrape_interval": "Cada cuánto Prometheus pide métricas",
    "evaluation_interval": "Cada cuánto evalúa reglas de alert",
    "metrics_path": "Path donde app expone /metrics",
    "targets": "Lista de hosts a scrapear",
}
for k, v in conceptos.items():
    print(f"  {k}: {v}")

## Histogram Buckets y Latencia

> Los buckets de histogram capturan cuántas observaciones cayeron en cada rango. Esto permite calcular percentiles con rate() y quantile().

In [ ]:
print("=== Histogram Buckets ===")

print("""
Buckets típicos para latencia HTTP (en segundos):
  http_request_duration_seconds_bucket{le='0.005'}    1000
  http_request_duration_seconds_bucket{le='0.01'}     2500
  http_request_duration_seconds_bucket{le='0.025'}    6000
  http_request_duration_seconds_bucket{le='0.05'}     9000
  http_request_duration_seconds_bucket{le='0.1'}      9800
  http_request_duration_seconds_bucket{le='0.25'}    10000
  http_request_duration_seconds_bucket{le='1.0'}     10000
  http_request_duration_seconds_bucket{le='+Inf'}     10000

Cálculo de percentiles:
  p50 = histogram_quantile(0.50, rate(http_request_duration_seconds_bucket[5m]))
  p90 = histogram_quantile(0.90, rate(http_request_duration_seconds_bucket[5m]))
  p99 = histogram_quantile(0.99, rate(http_request_duration_seconds_bucket[5m]))

+Inf siempre tiene el total de requests.
""")

## Dashboards en Grafana

> Grafana se conecta a Prometheus como datasource y permite crear visualizaciones. Paneles típicos: rate de requests, latencia p99, uso de memoria, errores por segundo.

In [ ]:
print("=== Dashboards Grafana ===")

paneles = {
    "Requests por segundo": "sum(rate(http_requests_total[1m])) by (method)",
    "Latencia p99": "histogram_quantile(0.99, rate(http_request_duration_seconds[5m]))",
    "Tasa de errores 5xx": "sum(rate(http_requests_total{status=~'5..'}[1m]))",
    "CPU usage": "process_cpu_seconds_total * 100",
    "Memoria en uso": "process_resident_memory_bytes",
    "Conexiones activas": "sum by (service) (active_connections)",
}

print("Paneles típicos de Grafana con PromQL:\n")
for nombre, consulta in paneles.items():
    print(f"{nombre}:")
    print(f"  {consulta}\n")

print("""
Configurar datasource en Grafana:
  1. Settings > Data Sources > Add data source
  2. Seleccionar Prometheus
  3. URL: http://prometheus:9090
  4. Save & Test

Crear dashboard:
  1. + > Dashboard > New panel
  2. Escribir PromQL en поле query
  3. Seleccionar tipo de visualización
  4. Guardar dashboard
""")

## Tips y Mejores Prácticas

> Usa labels liberally — es fácil agregar labels pero imposible removerlos sin romper backward compatibility.

> Para latencia, usa siempre Histogram, no Summary. Histogram permite agregar percentiles en Prometheus; Summary los calcula en la aplicación.

> No uses labels con cardinalidad alta (user_id, session_id) — esto puede causar memoria explosiva en Prometheus.

> Normaliza los nombres de métricas: usa snake_case y prefijos como namespace_producto (ej: miapp_http_requests_total).

> Incluye labels de servicio y versión para poder filtrar cuando tienes múltiples instancias o versiones.

## Errores Comunes

### Métricas sin labels

¿Por qué ocurre?
- Se crea una métrica sin labels y luego no se puede segmentar.

Solución
- Define labels desde el inicio: method, endpoint, status, service.

### Cardinalidad excesiva

¿Por qué ocurre?
- Se usan valores únicos como user_id o session_id como labels.

Solución
- Prometheus tiene límite de label cardinality. Usa labels estáticos, no valores únicos.

### Histogram en vez de Summary

¿Por qué ocurre?
- Summary calcula percentiles en la app, Prometheus no puede re-agregarlos.

Solución
- Usa Histogram siempre. Permite agregar datos en Prometheus para análisis multi-servicio.

### No exponer /metrics

¿Por qué ocurre?
- La aplicación no tiene el endpoint /metrics configurado.

Solución
- Añade route /metrics con generate_latest(). Usa librería según tu framework.

### Scrape interval muy largo

¿Por qué ocurre?
- scrape_interval: 1h para métricas de alta frecuencia.

Solución
- Para rate() preciso, usa scrape_interval <= 1/4 del rango de rate().